# Phase 3: Flight & Hotel Agents with Dynamic Mock APIs

Replace stub flight and hotel agents with real LLM-powered agents using:
- **Claude** (ChatAnthropic) for reasoning
- **Dynamic mock APIs** that generate realistic data for ANY destination
- **ReAct loop** (reason -> tool call -> observe -> reason)
- **LangGraph ToolNode** for tool execution

### Tools introduced:
- `search_flights` — generates route-based flight options with real airlines, plausible prices
- `get_airport_code` — city to IATA code lookup (50+ airports)
- `search_hotels` — generates tiered hotel options (luxury/upscale/midrange) with region pricing
- `get_hotel_reviews` — generates realistic guest reviews

## 1. Setup

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"LLM Provider: {os.getenv('LLM_PROVIDER', 'anthropic')}")

## 2. Test Flight Tools

Flight tools use **dynamic generation** — data is seeded by route so results are deterministic but realistic for any origin/destination pair.

In [ ]:
from src.tools.flight_tools import search_flights, get_airport_code
import json

# Test airport code lookups
print("=== Airport Code Lookups ===")
for city in ["New York", "Bali", "Paris", "Tokyo", "Santorini", "Luang Prabang"]:
    result = json.loads(get_airport_code.invoke(city))
    print(f"  {city:20s} -> {result['code']} ({result['name']})")
print()

In [ ]:
# Test flight search for multiple routes
routes = [
    ("New York", "Bali", 2),
    ("Chicago", "Maldives", 2),
    ("San Francisco", "Santorini", 2),
    ("Miami", "Cancun", 4),
]

for origin, dest, pax in routes:
    print(f"=== {origin} -> {dest} ({pax} travelers) ===")
    result = json.loads(search_flights.invoke({"origin": origin, "destination": dest, "travelers": pax}))
    for f in result["flights"]:
        print(f"  {f['airline']:25s} | {f['route']:25s} | {f['duration']:10s} | "
              f"${f['price_per_person']:>6}/pp | ${f['total_price']:>7} total | "
              f"{f['layovers']} stop(s) | Rating: {f['rating']}")
    print()

## 3. Test Hotel Tools

Hotel tools generate options across price tiers, with region-aware pricing and realistic amenities.

In [ ]:
from src.tools.hotel_tools import search_hotels, get_hotel_reviews

# Test hotel search for multiple destinations
destinations = [
    ("Bali", 5, 150),
    ("Paris", 4, 250),
    ("Santorini", 7, 0),
    ("Cancun", 5, 100),
]

for dest, nights, budget in destinations:
    budget_str = f"budget ${budget}/night" if budget > 0 else "no budget filter"
    print(f"=== {dest} ({nights} nights, {budget_str}) ===")
    result = json.loads(search_hotels.invoke({"destination": dest, "nights": nights, "budget_per_night": budget}))
    for h in result["hotels"]:
        print(f"  {h['name']:35s} | {h['star_rating']}* | {h['location']:20s} | "
              f"${h['price_per_night']:>4}/night | ${h['total_price']:>6} total | "
              f"Rating: {h['guest_rating']}/10")
    print()

In [ ]:
# Test hotel reviews
print("=== Reviews: Four Seasons Bali ===")
reviews = json.loads(get_hotel_reviews.invoke({"hotel_name": "Four Seasons Bali", "destination": "Bali"}))
print(f"Overall: {reviews['overall_rating']}/10 ({reviews['review_count']} reviews)")
for r in reviews["sample_reviews"]:
    print(f"  [{r['rating']}/10] {r['text']}")
print()

print("=== Reviews: Holiday Inn Paris ===")
reviews = json.loads(get_hotel_reviews.invoke({"hotel_name": "Holiday Inn Paris", "destination": "Paris"}))
print(f"Overall: {reviews['overall_rating']}/10 ({reviews['review_count']} reviews)")
for r in reviews["sample_reviews"]:
    print(f"  [{r['rating']}/10] {r['text']}")

## 4. Test Flight Agent (ReAct with Tools)

The flight agent uses Claude to reason about the route, calls tools for data, then synthesizes a ranked comparison.

In [ ]:
from src.agents.flight_agent import _run_react_agent as flight_react

print("=== Flight Agent: New York -> Bali (2 travelers) ===")
print()
result = flight_react("New York", "Bali", 2)
print(result)

In [ ]:
print("=== Flight Agent: San Francisco -> Santorini (2 travelers) ===")
print()
result = flight_react("San Francisco", "Santorini", 2)
print(result)

## 5. Test Hotel Agent (ReAct with Tools)

The hotel agent uses Claude to search hotels, check reviews, and produce a ranked recommendation.

In [ ]:
from src.agents.hotel_agent import _run_react_agent as hotel_react

print("=== Hotel Agent: Bali (5 nights, ~$150/night budget) ===")
print()
result = hotel_react("Bali", 5, 150)
print(result)

In [ ]:
print("=== Hotel Agent: Santorini (7 nights, no budget filter) ===")
print()
result = hotel_react("Santorini", 7, 0)
print(result)

## 6. Run the Full 3-Agent Pipeline

Destination agent uses Claude + DuckDuckGo (Phase 2). Flight and hotel agents use Claude + dynamic mock APIs (Phase 3). Activity and budget agents remain stubs (Phase 4).

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from src.state import TravelPlanState
from src.supervisor import parse_intent, supervisor_router, aggregate_results
from src.agents.destination_agent import destination_research_agent
from src.agents.flight_agent import flight_search_agent
from src.agents.hotel_agent import hotel_search_agent
from src.agents.activity_agent import activity_itinerary_agent
from src.agents.budget_agent import budget_currency_agent

builder = StateGraph(TravelPlanState)
builder.add_node("intent_parser", parse_intent)
builder.add_node("destination_research", destination_research_agent)
builder.add_node("flight_search", flight_search_agent)
builder.add_node("hotel_search", hotel_search_agent)
builder.add_node("activity_itinerary", activity_itinerary_agent)
builder.add_node("budget_currency", budget_currency_agent)
builder.add_node("aggregator", aggregate_results)

builder.add_edge(START, "intent_parser")
builder.add_conditional_edges("intent_parser", supervisor_router)
builder.add_conditional_edges("destination_research", supervisor_router)
builder.add_conditional_edges("flight_search", supervisor_router)
builder.add_conditional_edges("hotel_search", supervisor_router)
builder.add_conditional_edges("activity_itinerary", supervisor_router)
builder.add_conditional_edges("budget_currency", supervisor_router)
builder.add_edge("aggregator", END)

graph = builder.compile()
print("Graph compiled with REAL destination + flight + hotel agents!")
print()

# Run with Bali request
print("Running full graph...")
print()
for step in graph.stream({
    "messages": [
        HumanMessage(
            content="Plan a 5-day beach vacation in Bali for 2 people, "
            "budget $3000, departing from New York. Dates: March 15-20, 2025."
        )
    ],
}):
    node_name = list(step.keys())[0]
    node_output = step[node_name]
    agent = node_output.get('current_agent', '')

    real_agents = {'destination_research', 'flight_search', 'hotel_search'}
    if node_name in real_agents:
        label = '(REAL - LLM + Tools)'
    elif node_name in ('intent_parser', 'aggregator'):
        label = ''
    else:
        label = '(stub)'

    print(f"--- {node_name} {label} ---")
    if 'messages' in node_output and node_output['messages']:
        content = node_output['messages'][-1].content
        if len(content) > 400:
            print(f"  {content[:400]}...")
        else:
            print(f"  {content}")
    print()

## 7. Test with a Different Destination

Verify dynamic generation works for destinations not hardcoded in Phase 1 stubs.

In [ ]:
print("Running full graph for Santorini...")
print()
for step in graph.stream({
    "messages": [
        HumanMessage(
            content="Plan a 7-day romantic getaway to Santorini for 2 people, "
            "budget $5000, departing from San Francisco. Dates: June 10-17, 2025."
        )
    ],
}):
    node_name = list(step.keys())[0]
    node_output = step[node_name]

    real_agents = {'destination_research', 'flight_search', 'hotel_search'}
    if node_name in real_agents:
        label = '(REAL - LLM + Tools)'
    elif node_name in ('intent_parser', 'aggregator'):
        label = ''
    else:
        label = '(stub)'

    print(f"--- {node_name} {label} ---")
    if 'messages' in node_output and node_output['messages']:
        content = node_output['messages'][-1].content
        if len(content) > 400:
            print(f"  {content[:400]}...")
        else:
            print(f"  {content}")
    print()

## 8. Inspect Structured State

Run the graph and inspect the structured flight and hotel data in the final state.

In [ ]:
import json

final_state = graph.invoke({
    "messages": [
        HumanMessage(
            content="Plan a 5-day trip to Cancun for 4 people, "
            "budget $4000, departing from Miami. Dates: December 20-25, 2025."
        )
    ],
})

print("=== Flight Options (structured) ===")
flights = final_state.get("flight_options", {})
print(f"Source: {flights.get('source', 'unknown')}")
print(f"Recommended: {flights.get('recommended', 'N/A')}")
print(f"Cheapest: {flights.get('cheapest', 'N/A')}")
for i, f in enumerate(flights.get('options', []), 1):
    print(f"  {i}. {f['airline']} ({f['route']}) - ${f['total_price']} total | {f['duration']}")
print()

print("=== Hotel Options (structured) ===")
hotels = final_state.get("hotel_options", {})
print(f"Source: {hotels.get('source', 'unknown')}")
print(f"Nights: {hotels.get('nights', 'N/A')}")
print(f"Recommended: {hotels.get('recommended', 'N/A')}")
for i, h in enumerate(hotels.get('options', []), 1):
    print(f"  {i}. {h['name']} ({h['star_rating']}*) - ${h['total_price']} total | "
          f"{h['location']} | Rating: {h['guest_rating']}/10")

## Summary

Phase 3 complete:
- **search_flights** tool with dynamic mock API — generates realistic flights for ANY origin/destination
- **get_airport_code** tool — 50+ airport IATA code lookups
- **search_hotels** tool with dynamic mock API — tiered options with region-aware pricing
- **get_hotel_reviews** tool — generates plausible guest reviews
- **Flight ReAct agent** using Claude + LangGraph ToolNode
- **Hotel ReAct agent** using Claude + LangGraph ToolNode (budget-aware from flight costs)
- **Fallback** to dynamic mock data if LLM API is unavailable
- Tested with Bali, Santorini, and Cancun (all dynamically generated)

### Key design decisions:
- **Dynamic generation over hardcoded data** — uses hash-seeded randomness so results are deterministic per route but realistic for any city pair
- **Region-based pricing** — Southeast Asia is cheaper than Europe, Indian Ocean is premium
- **Hotel budget awareness** — hotel agent subtracts estimated flight cost from total budget to set a realistic per-night target

Next: Phase 4 — Activity/Itinerary & Budget/Currency agents.